# Complete Colab preparation

Formal DiffJev workflow; see AGENTS.md and prompt.md.

## 1. Choose preparation actions
This is the single preparation entry point. It mounts Drive, clones or reuses your checkout, installs an isolated Python 3.12 environment, downloads selected pinned assets, restores local withheld benchmark states, and prepares frozen data/scaling manifests. It never trains a model or calls a paid API. Set the three action switches to True for a first complete setup. Leave them False to inspect/reuse existing assets. The Colab notebook kernel may be Python 3.13; research commands use the separate Python 3.12 worker.

In [11]:
import os, sys, subprocess, json
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/BoldHu/DiffJev.git"
# Optional exact code commit for a NEW clone; an existing checkout is never reset.
CODE_REVISION = None
MOUNT_DRIVE = IN_COLAB
local_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_DIR = "/content/drive/MyDrive/DiffJev" if IN_COLAB else str(local_root)
ENV_DIR = "/content/diffjev-env" if IN_COLAB else str(local_root / ".venv-formal")

INSTALL_ENV = True
DOWNLOAD_ASSETS = True
PREPARE_DATA = True

# Start with the anchor. Add panel entries when their GPU/storage runs are ready.
TRAIN_BACKBONES = ["llada_8b_base"]
# Full panel: ["llada_8b_base", "nemotron_3b", "nemotron_8b", "nemotron_14b"]
DOWNLOAD_REFERENCE_BASELINES = False  # Jeva, Qwen AR reference, BGE
EXTRA_DATASETS = []  # Only add fallback_choice/noul/score if an audit needs one.
SEED = 0
SCALING_QUERIES = 512
RESTORE_WITHHELD_STATES = True
DOWNLOAD_BENCHMARK_SOURCES = True  # Only runs with DOWNLOAD_ASSETS and PREPARE_DATA.
SOURCE_ROOT = None  # default: <project>/data/raw/jevbench_sources


## 2. Mount storage and locate the checkout
Use PROJECT_DIR for the uploaded repository. Otherwise this cell clones it. Existing code is reused without pull/reset, so upload the revised notebook AND its scripts/src files before running. Drive stores assets, frozen manifests, checkpoints and logs; the Python environment is on the temporary Colab disk and must be recreated after a runtime reset.

In [12]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
ROOT = Path(PROJECT_DIR).expanduser().resolve()
if not (ROOT / "AGENTS.md").exists():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError("PROJECT_DIR is nonempty but is not this project; choose the uploaded checkout path")
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    if CODE_REVISION:
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
for required in ["src/diffjev/notebook_runtime.py", "scripts/prepare_workspace.py", "requirements-lock.txt"]:
    if not (ROOT / required).is_file():
        raise FileNotFoundError(f"Missing {required}; upload the updated project files, not only this notebook")
os.chdir(ROOT)
os.environ["DIFFJEV_ROOT"] = str(ROOT)
# Keep utility download paths consistent with the formal checkout-relative paths.
for key,folder in [("DIFFJEV_MODELS_DIR","models"),("DIFFJEV_DATA_DIR","data"),("DIFFJEV_OUTPUT_DIR","outputs")]:
    os.environ[key] = str(ROOT / folder)
sys.path.insert(0, str(ROOT / "src"))
from diffjev.notebook_runtime import NotebookRuntime, install_environment
print("Notebook kernel:", sys.version.split()[0])
print("Persistent project:", ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook kernel: 3.13.16
Persistent project: /content/drive/MyDrive/DiffJev


## 3. Install the pinned worker environment
INSTALL_ENV=True downloads Python 3.12 if necessary and installs requirements-lock.txt, including the official Jeva runtime. This does not replace Colab's kernel, so no kernel restart is required. Repeated installation reuses downloaded packages. Allow enough local disk for the environment and its package cache. No model weights or datasets are downloaded by this cell.

In [13]:
if INSTALL_ENV:
    runtime = install_environment(ROOT, ENV_DIR)
else:
    worker = Path(ENV_DIR) / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    runtime = NotebookRuntime(ROOT, worker)
    if not worker.exists():
        raise FileNotFoundError("Worker environment missing. Set INSTALL_ENV=True in cell 1 and rerun from there.")
    runtime.save_settings()
runtime.run([runtime.python, "scripts/prepare_workspace.py", "check"], "check_environment")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 30.7 MB/s  0:00:00
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /content/diffjev-env
Activate with: source /content/diffjev-env/bin/activate
Using Python 3.12.3 environment at: /content/diffjev-env
Resolved 93 packages in 1.02s
Prepared 93 packages in 37.75s
Installed 93 packages in 224ms
 + accelerate==1.12.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + anyio==4.16.0
 + attrs==26.1.0
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + contourpy==1.4.0
 + cycler==0.12.1
 + datasets==4.4.1
 + dill==0.4.0
 + fastjsonschema==2.22.3
 + filelock==4.2.9
 + fonttools==4.68.0
 + frozenlist==1.8.0
 + fsspec==2025.10.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.5.0
 + idna==3.20
 + iniconfig==2.3.1
 + jeva-model==0.1.0 (from git+https://github.com/LiQiiiii/Jeva.git@e2104e0720100f979f2cdd456a1cbbee96

## 4. Save the run selection and inspect the asset plan
All four backbone tokenizers/configs are needed for the shared context audit. The downloader now fetches those WITHOUT weights. Full weights are downloaded only for TRAIN_BACKBONES plus optional reference baselines. The five datasets include OpenJev SFT, ToolRet training, original JevBench cases, and held-out ToolRet Queries/Tools. No fallback dataset is added by default. The original release may withhold licensed benchmark state text; its manifest download is not a substitute for that text.

In [14]:
CORE_MODELS = ["llada_8b_base", "nemotron_3b", "nemotron_8b", "nemotron_14b"]
if not TRAIN_BACKBONES or any(name not in CORE_MODELS for name in TRAIN_BACKBONES):
    raise ValueError("Choose one or more required backbone keys")
FULL_MODELS = list(TRAIN_BACKBONES)
if DOWNLOAD_REFERENCE_BASELINES:
    FULL_MODELS += ["jeva_4b", "llm_as_jev_qwen", "bge_large_en"]
DATASETS = ["openjev", "toolret", "jevbench", "toolret_queries", "toolret_tools"] + EXTRA_DATASETS
plan = dict(seed=SEED, backbones=TRAIN_BACKBONES, full_models=FULL_MODELS,
            datasets=DATASETS, scaling_queries=SCALING_QUERIES)
plan_path = ROOT / "outputs/setup/plan.json"
plan_path.write_text(json.dumps(plan, indent=2) + "\n", encoding="utf-8")
runtime.python_code("""from pathlib import Path
import json, yaml
plan=json.loads(Path('outputs/setup/plan.json').read_text())
path=Path('configs/workflow.yaml')
cfg=yaml.safe_load(path.read_text())
cfg.update(seed=plan['seed'],backbones=plan['backbones'],scaling_queries=plan['scaling_queries'])
path.write_text(yaml.safe_dump(cfg,sort_keys=False))
print('Notebook run selection:',cfg['backbones'])
""", "save_run_selection")
print(json.dumps(plan, indent=2))
runtime.run([runtime.python, "scripts/download_assets.py", "--list"], "asset_plan")


Notebook run selection: ['llada_8b_base']
{
  "seed": 0,
  "backbones": [
    "llada_8b_base"
  ],
  "full_models": [
    "llada_8b_base"
  ],
  "datasets": [
    "openjev",
    "toolret",
    "jevbench",
    "toolret_queries",
    "toolret_tools"
  ],
  "scaling_queries": 512
}
=== Configured Pretrained & Baseline Models ===
  llada_8b_base      | repo=GSAI-ML/LLaDA-8B-Base                  | rev=0f2787f2d87eac5eed8a087d5ecd24277e6255b2 | path=/content/drive/MyDrive/DiffJev/models/GSAI-ML__LLaDA-8B-Base
  nemotron_3b        | repo=nvidia/Nemotron-Labs-Diffusion-3B      | rev=0d51902da1f8869f83413ce642fab402fa5641e0 | path=/content/drive/MyDrive/DiffJev/models/nvidia__Nemotron-Labs-Diffusion-3B
  nemotron_8b        | repo=nvidia/Nemotron-Labs-Diffusion-8B      | rev=16c67f0560b912e93e0cabb6e0c4f5c3086d95fc | path=/content/drive/MyDrive/DiffJev/models/nvidia__Nemotron-Labs-Diffusion-8B
  nemotron_14b       | repo=nvidia/Nemotron-Labs-Diffusion-14B     | rev=f8c3e2c078e193599b8882d965b10

## 5. Download selected pinned assets
Set DOWNLOAD_ASSETS=True to execute this cell's downloads. Re-running resumes/reuses snapshot files. Downloads require sufficient Drive quota and Colab disk; full panel weights alone are tens of GB. Existing Hugging Face authentication is used if configured; do not paste tokens into notebook cells. Jev API remains disabled.

In [15]:
def download_arguments(option, names):
    return [part for name in names for part in [option, name]]
if DOWNLOAD_ASSETS:
    runtime.run([runtime.python, "scripts/download_assets.py", "--metadata-only", "--yes"]
                + download_arguments("--model", CORE_MODELS), "download_metadata")
    runtime.run([runtime.python, "scripts/download_assets.py", "--yes"]
                + download_arguments("--model", FULL_MODELS), "download_weights")
    runtime.run([runtime.python, "scripts/download_assets.py", "--yes"]
                + download_arguments("--dataset", DATASETS), "download_datasets")
else:
    print("Downloads disabled; existing local snapshots will be checked.")



Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 20.10it/s]
/content/drive/MyDrive/DiffJev/scripts/download_assets.py:94: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "downloaded_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
[model:llada_8b_base] saved to /content/drive/MyDrive/DiffJev/models/GSAI-ML__LLaDA-8B-Base

Fetching 9 files: 100%|██████████| 9/9 [00:01<00:00,  8.09it/s]
[model:nemotron_3b] saved to /content/drive/MyDrive/DiffJev/models/nvidia__Nemotron-Labs-Diffusion-3B

Fetching 9 files: 100%|██████████| 9/9 [00:01<00:00,  5.80it/s]
[model:nemotron_8b] saved to /content/drive/MyDrive/DiffJev/models/nvidia__Nemotron

## 6. Restore benchmark text and prepare the shared data
Set PREPARE_DATA=True. This runs the existing adapters, cross-source overlap audit, 90/5/5 splits, training-only banks, 3,125-update augmentation trace, and fixed 512-query random/BM25-hard scaling manifests. It is a substantial CPU/RAM operation, not training. Existing complete manifests are verified and reused; partial/mismatched preparation fails rather than silently changing comparisons.

With DOWNLOAD_ASSETS and DOWNLOAD_BENCHMARK_SOURCES enabled, missing upstream files needed for withheld states (including SST5 test.jsonl) are fetched at the exact revisions from sources.json. Alternatively upload original sources under data/raw/jevbench_sources or supply restored_states.jsonl. Restoration verifies state hashes. Missing licensed sources or a needed fallback partition produces an explicit error in this cell; supply those assets and rerun. Review audit.json before freezing experiments.

In [16]:
if PREPARE_DATA:
    args = [runtime.python, "scripts/prepare_workspace.py", "prepare", "--seed", str(SEED),
            "--scaling-queries", str(SCALING_QUERIES)]
    if RESTORE_WITHHELD_STATES:
        args.append("--restore-states")
        if DOWNLOAD_ASSETS and DOWNLOAD_BENCHMARK_SOURCES:
            args.append("--download-benchmark-sources")
    if SOURCE_ROOT:
        args += ["--source-root", str(SOURCE_ROOT)]
    runtime.run(args, "prepare_data")
else:
    print("Preparation disabled; enable PREPARE_DATA for a first run.")


Traceback (most recent call last):
  File "/content/drive/MyDrive/DiffJev/scripts/prepare_workspace.py", line 27, in <module>
    main()
  File "/content/drive/MyDrive/DiffJev/scripts/prepare_workspace.py", line 22, in main
    prepare_all(args.seed,args.scaling_queries,args.restore_states,args.source_root)
  File "/content/drive/MyDrive/DiffJev/src/diffjev/preparation.py", line 76, in prepare_all
    print(w.restore_benchmark_states(source_root))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/DiffJev/src/diffjev/workflows.py", line 217, in restore_benchmark_states
    sources[source]=common.load_rows(common.source_by_key(source))
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/DiffJev/third_party/jevbench/common.py", line 104, in load_rows
    raise FileNotFoundError(f"{path} is missing. Run workspace/download.py --only {src['key']} first.")
FileNotFoundError: /content/drive/MyDrive/DiffJev/data/raw/je

RuntimeError: prepare_data failed (exit 1); see /content/drive/MyDrive/DiffJev/outputs/setup/logs/prepare_data.log

## 7. Readiness report and next notebook
The report distinguishes tokenizer/config presence from full weight files and prepared data. It does not assert real-weight parity or GPU fit. Logs stream to outputs/setup/logs; environment, preparation and readiness reports are saved next to them. Once preparation is complete, notebook 01 is optional for rechecking/auditing. Start training in 02. Notebooks 01–07 use the same saved Python worker, so Colab's default Python/package versions do not affect formal execution. Enable reference downloads before running notebook 03's reference-model cells, and fetch the full panel before 07.

In [ ]:
runtime.run([runtime.python, "scripts/prepare_workspace.py", "status", "--seed", str(SEED)]
            + download_arguments("--model", FULL_MODELS), "readiness")
print("Read the report:", ROOT / "outputs/setup/readiness.json")
print("After readiness and audit review, open notebooks/02_train_diffjev.ipynb.")
